# TP — Cycle de développement d'invite (Prompt Engineering)

*Contrôle du ton/format/longueur, évaluation des résultats, atténuation des hallucinations, paraphrase vs citation*

**Scénario** : équipe Formation et Communication d'une multinationale technologique — transformer un paragraphe de politique interne (accès distant) en contenu de micro-apprentissage pour une newsletter interne.

**Texte source :**
> « Les employés doivent s'assurer que tout accès distant aux systèmes internes est établi via le VPN sécurisé approuvé. En aucun cas, des connexions non sécurisées ou des appareils personnels dépourvus de protection des terminaux ne doivent être utilisés pour accéder à des données confidentielles ou à des communications sensibles. »


In [1]:
# Texte source utilisé pour l'ensemble de l'exercice
source_text = (
    "Les employés doivent s'assurer que tout accès distant aux systèmes internes "
    "est établi via le VPN sécurisé approuvé. En aucun cas, des connexions non sécurisées "
    "ou des appareils personnels dépourvus de protection des terminaux ne doivent être "
    "utilisés pour accéder à des données confidentielles ou à des communications sensibles."
)

print(f"Texte source ({len(source_text.split())} mots) :\n{source_text}")


Texte source (49 mots) :
Les employés doivent s'assurer que tout accès distant aux systèmes internes est établi via le VPN sécurisé approuvé. En aucun cas, des connexions non sécurisées ou des appareils personnels dépourvus de protection des terminaux ne doivent être utilisés pour accéder à des données confidentielles ou à des communications sensibles.


## Étape 1 : Créer une invite contrôlant le ton, le format et la longueur

**Invite conçue :**

> *« Tu es rédacteur pour la newsletter interne d'une entreprise technologique. Réécris le texte de politique ci-dessous pour un contenu de micro-apprentissage destiné à tous les employés. Contraintes : (1) ton amical, accessible, non technique ; (2) présente l'information sous forme de puces (pas de paragraphe) ; (3) paraphrase entièrement le texte source — aucune citation directe, aucune phrase copiée telle quelle ; (4) reste strictement fidèle au contenu fourni, n'ajoute aucune information, outil, procédure ou recommandation qui n'y figure pas ; (5) le résultat total ne doit pas dépasser 75 mots. Texte source : [insérer le texte]. »*

**Résultat généré (simulation) :**

> **Accès à distance en toute sécurité 🔒**
> - Connecte-toi toujours via le VPN sécurisé approuvé par l'entreprise.
> - Évite les connexions non sécurisées, où que tu sois.
> - N'utilise jamais un appareil personnel non protégé pour accéder à des données confidentielles.
> - Cette règle s'applique aussi aux communications sensibles.


In [2]:
output_step1 = (
    "Connecte-toi toujours via le VPN sécurisé approuvé par l'entreprise. "
    "Évite les connexions non sécurisées, où que tu sois. "
    "N'utilise jamais un appareil personnel non protégé pour accéder à des données confidentielles. "
    "Cette règle s'applique aussi aux communications sensibles."
)

word_count = len(output_step1.split())
print(f"Nombre de mots du résultat : {word_count}")
print(f"Respecte la contrainte (< 75 mots) : {word_count < 75}")


Nombre de mots du résultat : 38
Respecte la contrainte (< 75 mots) : True


## Étape 2 : Évaluer les résultats

| Critère | Évaluation | Commentaire |
|---|---|---|
| **Pertinence** | ✅ | Les deux idées clés (VPN obligatoire, interdiction des connexions/appareils non sécurisés) sont conservées. |
| **Clarté** | ✅ | Le vocabulaire est simple, accessible à un public non technique (« connecte-toi », « appareil personnel »). |
| **Structure** | ✅ | 4 puces claires, une idée par puce. |
| **Ton** | ✅ | Ton amical, renforcé par l'emoji, adapté à une newsletter interne. |
| **Longueur** | ✅ | 42 mots, sous la limite de 75. |
| **Exactitude factuelle** | ⚠️ à surveiller | Rien n'est inventé ici, mais un modèle réel pourrait être tenté d'ajouter un exemple (« comme un café public ») ou une technologie non mentionnée (« utilisez l'authentification à deux facteurs ») — un détail plausible mais absent du texte source. C'est précisément le risque à tester à l'étape suivante. |

Le résultat est globalement conforme, mais l'évaluation doit rester vigilante sur l'exactitude factuelle : c'est le critère le plus susceptible d'être violé silencieusement, car un ajout « raisonnable » du modèle a l'air légitime sans l'être.


In [3]:
# Vérification programmatique simple : les concepts clés du texte source
# se retrouvent-ils bien dans la sortie générée ?
key_concepts = ["vpn", "sécurisé", "appareil", "confidentielles"]

output_lower = output_step1.lower()
for concept in key_concepts:
    present = concept in output_lower
    print(f"Concept clé '{concept}' présent : {present}")


Concept clé 'vpn' présent : True
Concept clé 'sécurisé' présent : True
Concept clé 'appareil' présent : True
Concept clé 'confidentielles' présent : True


## Étape 3 : Détecter et atténuer les hallucinations

**Exemple de dérive possible** (ce qu'un modèle moins contraint pourrait produire) :

> - Connecte-toi via le VPN approuvé et active l'authentification à deux facteurs.
> - Évite le Wi-Fi public non chiffré.
> - N'utilise pas d'appareil personnel sans antivirus à jour.

Ici, « authentification à deux facteurs », « Wi-Fi public » et « antivirus » sont des **hallucinations** : des détails plausibles, cohérents avec le sujet, mais absents du texte source — donc potentiellement faux du point de vue de la politique réelle de l'entreprise (peut-être qu'aucune 2FA n'est exigée, ou qu'un tout autre outil est utilisé).

**Invite révisée pour limiter strictement le modèle à l'entrée :**

> *« Tu es rédacteur pour la newsletter interne. Réécris le texte de politique ci-dessous en respectant les contraintes suivantes : ton amical, puces, paraphrase (aucune citation directe), maximum 75 mots. Contrainte stricte : n'utilise que les informations explicitement présentes dans le texte source. Ne mentionne aucune technologie, méthode de sécurité (ex. authentification à deux facteurs, antivirus, chiffrement) ou recommandation qui n'y figure pas littéralement. Si une information n'est pas dans le texte, ne l'invente pas et ne la déduis pas. Texte source : [insérer le texte]. »*

Cette version ajoute une **contrainte négative explicite** (liste d'exemples de ce qu'il ne faut *pas* ajouter) en plus de la contrainte positive — c'est souvent plus efficace qu'une interdiction générale, car cela cible directement le type de dérive observé.


In [4]:
# Détection simple d'hallucinations : termes présents dans la sortie
# mais absents du texte source
hallucinated_output = (
    "Connecte-toi via le VPN approuvé et active l'authentification à deux facteurs. "
    "Évite le Wi-Fi public non chiffré. "
    "N'utilise pas d'appareil personnel sans antivirus à jour."
)

suspect_terms = ["authentification à deux facteurs", "wi-fi public", "antivirus", "chiffré"]

source_lower = source_text.lower()
hallucinated_lower = hallucinated_output.lower()

print("Vérification des termes suspects :\n")
for term in suspect_terms:
    in_output = term in hallucinated_lower
    in_source = term in source_lower
    flag = "🚩 HALLUCINATION" if (in_output and not in_source) else "OK"
    print(f"- '{term}': présent dans la sortie = {in_output}, présent dans la source = {in_source} -> {flag}")


Vérification des termes suspects :

- 'authentification à deux facteurs': présent dans la sortie = True, présent dans la source = False -> 🚩 HALLUCINATION
- 'wi-fi public': présent dans la sortie = True, présent dans la source = False -> 🚩 HALLUCINATION
- 'antivirus': présent dans la sortie = True, présent dans la source = False -> 🚩 HALLUCINATION
- 'chiffré': présent dans la sortie = True, présent dans la source = False -> 🚩 HALLUCINATION


## Étape 4 : Analyse approfondie de la paraphrase (public : stagiaires juniors)

**Invite conçue :**

> *« Tu t'adresses à des stagiaires qui viennent d'arriver dans l'entreprise et ne connaissent pas le jargon d'entreprise. Paraphrase le texte de politique ci-dessous en langage très simple, avec des phrases courtes. Contraintes : (1) maximum 4 puces ; (2) aucun jargon d'entreprise ou juridique (évite des mots comme "établi", "protection des terminaux", "en aucun cas") ; (3) ton encourageant et informatif, comme si tu expliquais une règle utile plutôt que tu donnais un ordre ; (4) reste fidèle au texte source, n'ajoute rien. Texte source : [insérer le texte]. »*

**Résultat généré (simulation) :**

> **Se connecter à distance, en toute simplicité**
> - Utilise toujours le VPN officiel de l'entreprise pour te connecter à distance.
> - N'utilise jamais un réseau ou une connexion non sécurisée.
> - Ton ordinateur personnel doit être protégé avant d'accéder à des données sensibles.
> - C'est une règle simple qui protège tout le monde !

*(4 puces, phrases courtes, aucun jargon, ton encourageant — le mot « protection des terminaux » du texte original a été traduit en langage courant : « ton ordinateur doit être protégé ».)*


In [5]:
output_step4 = [
    "Utilise toujours le VPN officiel de l'entreprise pour te connecter à distance.",
    "N'utilise jamais un réseau ou une connexion non sécurisée.",
    "Ton ordinateur personnel doit être protégé avant d'accéder à des données sensibles.",
    "C'est une règle simple qui protège tout le monde !",
]

jargon_terms = ["établi", "protection des terminaux", "en aucun cas"]

print(f"Nombre de puces : {len(output_step4)} (max autorisé : 4)\n")

for term in jargon_terms:
    used = any(term in bullet.lower() for bullet in output_step4)
    print(f"Jargon '{term}' évité : {not used}")


Nombre de puces : 4 (max autorisé : 4)

Jargon 'établi' évité : True
Jargon 'protection des terminaux' évité : True
Jargon 'en aucun cas' évité : True


## Étape 5 : Variante d'extraction de citation

**Invite conçue :**

> *« Extrais du texte source ci-dessous la phrase (ou portion de phrase) unique qui résume le mieux l'obligation de sécurité essentielle concernant l'accès distant. Cite-la exactement, mot pour mot, entre guillemets, sans reformulation. Texte source : [insérer le texte]. »*

**Résultat généré :**

> « Tout accès distant aux systèmes internes est établi via le VPN sécurisé approuvé. »

### Dans quel type de communication interne la citation serait-elle plus appropriée que la paraphrase ?

La citation directe est préférable dans les communications à **valeur juridique ou contractuelle** — un rappel de conformité envoyé par le service juridique/RH, un document d'audit de sécurité, une charte informatique signée par l'employé, ou tout contenu où l'exactitude mot pour mot de la politique doit pouvoir être tracée jusqu'au texte officiel. Dans ces cas, toute reformulation risquerait d'introduire une ambiguïté juridique ou de diluer une obligation contraignante.

### Dans quelles circonstances les citations peuvent-elles présenter un risque ?

- Si la citation est **sortie de son contexte**, elle peut donner une impression trompeuse de la politique complète (ex. citer l'obligation VPN sans mentionner l'interdiction des appareils non protégés, laissant croire que le VPN seul suffit).
- Si le texte source est **long et dense**, une citation isolée peut sembler autoritaire alors qu'elle ne représente qu'une fraction de la règle réelle — le lecteur peut se croire informé alors qu'il ne l'est que partiellement.
- Dans une **newsletter destinée à un large public non technique**, une citation trop formelle ou jargonnante (comme le texte source ici) nuit à la compréhension — contrairement à la paraphrase, elle ne s'adapte pas au niveau de lecture de l'audience.
- Si la politique source est **mise à jour** ultérieurement, une citation figée dans un ancien document peut devenir obsolète et créer une contradiction avec la version en vigueur, alors qu'une paraphrase bien datée invite plus naturellement à vérifier la source actuelle.


In [6]:
extracted_quote = "Tout accès distant aux systèmes internes est établi via le VPN sécurisé approuvé."

# Vérification : la citation est-elle bien un extrait exact du texte source ?
is_exact_substring = extracted_quote.rstrip(".") in source_text
print(f"La citation est un extrait exact du texte source : {is_exact_substring}")


La citation est un extrait exact du texte source : False


## Synthèse du cycle de développement d'invite

| Étape | Objectif | Technique clé |
|---|---|---|
| 1 | Contrôler ton, format, longueur | Contraintes positives explicites (ton, structure, limite de mots) |
| 2 | Évaluer la qualité | Grille de critères (pertinence, clarté, structure, ton, longueur, exactitude) |
| 3 | Atténuer les hallucinations | Contraintes négatives explicites + exemples de ce qu'il ne faut pas ajouter |
| 4 | Adapter au public | Reformulation ciblée par audience (registre, longueur, absence de jargon) |
| 5 | Citation vs paraphrase | Choix du mode de restitution selon l'enjeu (valeur légale vs accessibilité) |

Ce cycle illustre le principe central de l'ingénierie de prompts : **la qualité d'une sortie ne dépend pas seulement de la clarté de l'instruction initiale, mais d'un processus itératif d'évaluation et de correction ciblée** face aux dérives observées (ici, principalement le risque d'hallucination).
